# Week 2: Embeddings & Vector DB

This notebook demonstrates the full embedding pipeline built in `src/embeddings.py`:
- Load the resolved ticket corpus
- Build a ChromaDB collection (or load the existing one)
- Run example similarity searches
- Demo metadata filtering

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import chromadb
from src.embeddings import load_corpus, get_embedding_model, build_collection, search

CORPUS_PATH = "../data/processed/tickets_clean.csv"
CHROMA_PATH = "../data/chroma_db"
COLLECTION_NAME = "itsm_tickets"

## 2. Load Corpus

In [ ]:
df = load_corpus(CORPUS_PATH)
print(f"Corpus: {len(df)} resolved tickets")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

## 3. Load ChromaDB Collection

The collection was already built by running `python src/embeddings.py`. We just open the existing one here — no need to re-embed.

In [ ]:
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(COLLECTION_NAME)
model = get_embedding_model()

print(f"Collection '{COLLECTION_NAME}' — {collection.count()} tickets indexed")

## 4. Example Searches

Four queries covering different categories — each returns the top 3 most similar resolved tickets with their resolutions.

In [ ]:
queries = [
    "User cannot log into their laptop after a password reset",
    "Database connection keeps timing out during peak hours",
    "VPN drops connection every few minutes when working remotely",
    "New employee needs access to the HR shared drive",
]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 80)
    results = search(query, collection, model, top_k=3)
    for r in results:
        print(f"  [{r['category']}] {r['subject']} (dist: {r['distance']:.4f})")
        print(f"  Resolution: {r['resolution'][:120]}...")
        print()

## 5. Metadata Filtering

ChromaDB supports filtering by metadata fields before doing similarity search. Here we restrict results to a specific department.

In [ ]:
query = "cannot access shared files"

print("Without filter:")
for r in search(query, collection, model, top_k=3):
    print(f"  [{r['category']}] {r['subject']}")

print("\nFiltered to HR department only:")
for r in search(query, collection, model, top_k=3, where={"department": "HR"}):
    print(f"  [{r['category']}] {r['subject']}")